# Step 2 : 有限時間LQRの制御則とフィードバックゲイン

Step1 では入力列$U$を与えて、状態軌道$X$と総コスト$J$を順方向に計算を行った。これをrolloutと呼ぶ。

Step2 では、線形システムと二次コストからなる有限時間LQRを扱う。Bellman方程式を用いて価値関数を終端から逆向きに計算することで、各時刻の最適入力を状態から決める以下のフィードバックゲイン列を求める。

$$
K_0, K_1, \cdots , K_{N-1}
$$

このbackward pass で求めるのは、具体的な入力列ではなく、以下の最適制御則である。

$$
u_k^*(x_k) = -K_k \ x_k
$$

これは、具体的な入力値 $u_k^{*}$ を求めているのではなく、有限離散時間LQRにおいて$x_k$が与えられたときに必要となる制御則とそのフィードバックゲインは何かを求めているのである。

## 有限時間LQR で扱う問題

有限時間LQRでは線形システムを離散時間状態方程式として扱う。

$$
x_{k+1} = A_k x_k + B_k u_k
$$

コスト$J$は二次形式で次のようなものである。ここでは最初から離散時間で考えているため、ステージコスト$\ell_k$に$\Delta t$ は掛けていない。

$$
J = \phi(x_N) + \sum_{k=0}^{N-1} \ell_k ( x_k, u_k) = \frac{1}{2} {x_N}^T Q_N {x_N} + \sum_{k=0}^{N-1}\left( \frac{1}{2} {x_k}^T Q_k {x_k} + \frac{1}{2} {u_k}^T R_k {u_k} \right)
$$

ここでは簡単のため目標状態を$x_{ref}=0$ と原点としている。
- $Q_k$ : 状態を原点へ近づける重み (半正定値行列 $Q_k \succeq 0$)
- $R_k$ : 入力を小さくする重み (正定値行列 $R_k \succ 0$)
- $Q_N$ : 終端状態を原点へ近づける重み (半正定値行列 $Q_N \succeq 0$)

半正定値行列はその固有値が0以上であり、正定値行列はその固有値が正の値である行列である。例えば次のものである。

$$
Q_k = \begin{bmatrix}
1 & 0 \\
0 & 0
\end{bmatrix} , \quad
R_k = \begin{bmatrix}
1 & 0 \\
0 & 1
\end{bmatrix}
$$

これを制御的な意味で考えると、$Q_k \succeq 0$ によって評価しない状態方向があってもよい、であり、$R_k \succ 0$ によって入力は必ず評価するということになる。また後段の説明で、フィードバックゲインを求めるときにもこの条件が必要となる。

$\ell_k$ と表示しているが、これは $R_k, Q_k$ のように重み行列を時刻により変更することが出来るためである。
例えば終端に近づくほど目標誤差を重視するように設定することが出来る。

## 価値関数について

時刻 $k$ の状態 $x_k$ から終端までのコストを、残りの$U_k$ について最小化する。この時の最小コストが価値関数 $V_k(x_k)$ であり、それを実現する入力列が最適入力列 $U_k^*$ である。

$U_k$は以下のように$k$から$N-1$までの入力列である。

$$
U_k = \begin{bmatrix} u_k^T & u_{k+1}^T & \cdots & u_{N-1}^T \end{bmatrix}^T
$$

価値関数は$J_k$を$U_k$について最小化した後の最小コストである。($U_k$を変数として動かしたときに、最も小さい$J_k$の値)

$$
\begin{aligned}
V_k(x_k) &= \min\limits_{U_k} J_k(x_k, U_k) \\
& = \min\limits_{U_k} \left[ \sum_{i=k}^{N-1} \ell_i(x_i, u_i) + \phi(x_N) \right]
\end{aligned}
$$


一方、最小化すべき入力列は以下となる。($J_k$が最小となる入力列$U_k$)

$$
U_k^* = \underset{U_k}{\arg \min} J_k (x_k, U_k)
$$


## Bellmanの最適性原理

時刻 $k$ から終端$N$までの最小コストは、次の2つの合計を、現在の入力 $u_k$ について最小化することで求められる。

1. 現在の入力 $u_k$ によって発生するステージコスト $\ell_k(x_k, u_k)$
1. $u_k$によって決まる次の状態 $x_{k+1}$ から先の最小コスト $V_{k+1}(x_{k+1})$

$$
V_k(x_k) = \min\limits_{u_k} \left[ \ell_k(x_k, u_k) + V_{k+1}(x_{k+1})\right]
$$

$V_{k+1}$は入力列 $u_{k+1} , \cdots, u_{N-1}$について最小化済みだが、その出発点$x_{k+1}$によって値が変化する。

$$
V_{k+1}(x_{k+1}) = \min\limits_{u_{k+1}, \cdots, u_{N-1}} J_{k+1}(x_{k+1}, u_{k+1}, \cdots, u_{N-1})
$$

$x_{k+1}$は以下であるため、$u_k$によって変化する。

$$
x_{k+1} = A_k x_k + B_k u_k
$$

よって、現在の入力 $u_k$ を選ぶときには、現在のコスト $\ell_k$ と遷移先からの最小コスト $V_{k+1}$ の合計が最小になる $u_k$ を選ぶ必要がある。

$$
V_k(x_k) = \min\limits_{u_k} \left[ \ell_k(x_k, u_k) + V_{k+1}(A_k x_k + B_k u_k) \right]
$$

この式をBellman方程式と呼ぶ。

重要なことは、未来の入力列全部を一度に考える代わりに、今の入力$u_k$と、次の時刻以降の最小コスト $V_{k+1}$に分けていることである。

## 最適制御則

線形状態方程式と二次コストからなるLQRでは、Bellman方程式による最小化でも、価値関数は状態$x_k$について二次形式になる。以下では、その最小化を行うための最適制御則とそのためのRiccati再帰を導出する。

$$
V_{k}(x_{k}) = \frac{1}{2} {x_{k}}^T \ P_{k} \ {x_{k}}
$$

### 価値関数と最適入力列

時刻 $k$ の状態 $x_{k}$ を初期状態として固定する。そして、入力列 $U_{k}$ を仮に以下のように与える。

$$
U_{k} = [u_{k}^T, u_{k+1}^T, \cdots , u_{N-1}^T]^T
$$

状態方程式によって以下のように終端状態 $x_N$ まで順方向に計算することができる。これをrollout と呼ぶ。

$$
\begin{aligned}
x_{k+1} &= A_{k} \ x_{k} + B_{k} \ u_{k} \\
x_{k+2} &= A_{k+1} \ x_{k+1} + B_{k+1} \ u_{k+1} \\
& \ \vdots
\end{aligned}
$$

そして、その入力列$U_{k}$は以下のコスト $J_{k}$ によって評価することが出来る。

$$
J_{k} = \sum_{i=k}^{N-1} \ell_i(x_i, u_i) + \phi(x_N)
$$

同じ$x_{k}$から開始し、入力列を${U_{k}}^{(i)}$のように変化させれば状態軌道$X$とコスト$J$が変化する。

$$
\begin{aligned}
{U_{k}}^{(1)} &\rightarrow X^{(1)} \rightarrow {J_{k}}^{(1)} \\
{U_{k}}^{(2)} &\rightarrow X^{(2)} \rightarrow {J_{k}}^{(2)} \\
& \vdots
\end{aligned}
$$

これらの中から、$J_{k}$を最小化する入力列を $U_{k}^*$ と定義する。

$$
U_{k}^* = \underset{U_{k}}{\arg \min} J_{k} (x_{k}, U_{k})
$$

そして、その時の最小コストが価値関数$V_{k}(x_{k})$である。

$$
V_{k}(x_{k}) = \underset{U_{k}}{\min} J_{k} (x_{k}, U_{k})
$$

例えば、原点に戻す制御を考えた場合、

- $x_{k}$が原点の右側なら、左向きの入力が必要
- $x_{k}$が原点の左側なら、右向きの入力が必要
- $x_{k}$が原点にあるなら、入力は不要

となるため、最小コストを実現するための最適入力列$U_{k}^*$は、開始状態$x_{k}$によって変化する。したがって、最適入力列を初期状態$x_{k}$の関数として捉えることが出来る。

$$
U_{k}^* = U_{k}^*(x_{k})
$$

この最適入力列をコスト $J_{k}$に代入すると、$J_{k}$は最小化され価値関数は次のようになる。

$$
V_{k}(x_{k}) = J_{k} (x_{k}, U_{k}^*)
$$

最適入力列 $U_{k}^*$ は $x_{k}$ から決定されるため、結果的に価値関数は $x_{k}$ だけの関数となる。

$$
x_{k} \longrightarrow U_{k}^*(x_{k}) \longrightarrow V_{k}(x_{k}) \text{(最小コスト)}
$$


### 最適制御則とフィードバックゲイン列

最適入力を計算する最適制御則により価値関数$V_k$は$x_k$だけの関数として表すことが出来る。

$$
V_k(x_k) = \frac{1}{2} x_k^T P_k x_k
$$

これを$V_k(x_k)$の定義より考えていく。

$$
\begin{aligned}
V_k(x_k) &= \min\limits_{U_k} J_k(x_k, U_k) \\
& = \min\limits_{U_k} \left[ \sum_{i=k}^{N-1} \ell_i(x_i, u_i) + \phi(x_N) \right]
\end{aligned}
$$

$$
U_k = \begin{bmatrix} u_k^T & u_{k+1}^T & \cdots & u_{N-1}^T \end{bmatrix}^T
$$ 

まず、終端$N$では入力がないため、以下となる。

$$
V_N(x_N) = \phi(x_N) = \frac{1}{2}x_N^T Q_N x_N
$$

よって、$P_N = Q_N$ と置くことが出来る。

次に終端$N$のひとつ前の時刻 $N-1$では次となる。

$$
\begin{aligned}
V_{N-1}(x_{N-1}) &= \underset{u_{N-1}}{\min} J_{N-1} (x_{N-1}, u_{N-1}) \\
&= \min\limits_{u_{N-1}} \left[ \ell_{N-1}(x_{N-1}, u_{N-1})  + V_N(x_N) \right] \\
&= \underset{u_{N-1}}{\min} \left[ \frac{1}{2}x_{N-1}^T Q_{N-1} x_{N-1} + \frac{1}{2} u_{N-1}^T R_{N-1} u_{N-1} + \frac{1}{2}x_N^T P_N x_N \right]
\end{aligned}
$$

状態方程式より$x_N$は次のようになるため、これを $J_{N-1}$ に代入する。

$$
x_{N} = A_{N-1} x_{N-1} + B_{N-1} u_{N-1}
$$

$J_{N-1}$を示すと以下となり、$x_{N-1}$と$u_{N-1}$の二次形式になっている。

$$
\begin{aligned}
J_{N-1} = &\frac{1}{2}x_{N-1}^T Q_{N-1} x_{N-1} + \frac{1}{2} u_{N-1}^T R_{N-1} u_{N-1} \\
&+ \frac{1}{2} (A_{N-1} x_{N-1} + B_{N-1} u_{N-1})^T P_N (A_{N-1} x_{N-1} + B_{N-1} u_{N-1})
\end{aligned}
$$

$V_{N-1}(x_{N-1})$はその式が示すように、上式の二次形式を$u_{N-1}$について最小化する。

上式を$u_{N-1}$と$x_{N-1}$でまとめると以下となる。

$$
\begin{aligned}
J_{N-1} = &\frac{1}{2} u_{N-1}^T \ \left[ R_{N-1} + B_{N-1}^T P_N B_{N-1} \right] \ u_{N-1}  + u_{N-1}^T \left[ B_{N-1}^T P_N A_{N-1} \right] x_{N-1}\\
&+ \frac{1}{2} x_{N-1}^T \left[ Q_{N-1} + A_{N-1}^T P_N A_{N-1} \right] \ x_{N-1}
\end{aligned}
$$

$J_{N-1}$を$u_{N-1}$で最小化する前に、Hessianを調べて一意な最小点が存在することを確認する。

$J_{N-1}$を$u_{N-1}$で2階偏微分を行い Hessian $H$ を計算する。

$$
\frac{\partial^2 J_{N-1}}{\partial u_{N-1}^2} = H = R_{N-1} + B_{N-1}^T P_N B_{N-1}
$$

この$H$により関数の形は以下のように変化する。

- 正定値 $H \succ 0$ であればどの方向にも曲がるおわん型の狭義凸関数となり、一意な最小点を持つ。
- 半正定値 $H \succeq 0$ であれば平らな方向が存在する可能性があり、最小点が一意とは限らない。
- 不定 であれば、負にまがる方向が存在するため、一般にはコストは下に非有界となる、有限な最小点をもたない。

$P_N = Q_N \succeq 0$ であり、$R_{N-1} \succ 0 $ とコスト関数では設定している。そのため、任意のゼロでないベクトル$z$に対し、2次形式を計算すると、 $z^T H z > 0$ となり$H$が正定値であることを示している。

$$
\begin{aligned}
z^T H z &= z^T  R_{N-1} z + z^T B_{N-1}^T P_N B_{N-1} z\\
&= \underset{ > 0 }{\underbrace{z^T  R_{N-1} z }} + \underset{ \ge 0 }{\underbrace{  (B_{N-1}z)^T P_N (B_{N-1} z)}}\\
& > 0
\end{aligned}
$$

よって、$J_{N-1}$ は$u_{N-1}$について狭義凸であり、偏微分がゼロになる点が一意な最小点となる。

価値関数 $V_{N-1}(x_{N-1})$ を求めることは、コスト $J_{N-1}(x_{N-1}, u_{N-1})$ を $u_{N-1}$ について最小化することである。この最小化によって、価値関数$V_{N-1}$と、その最小コストを実現する最適制御則 $u_{N-1}^*$ を得ることができる。

$J_{N-1}$を$u_{N-1}$で偏微分してゼロとする。

$$
[R_{N-1} + B_{N-1}^T P_N B_{N-1}] u_{N-1} + B_{N-1}^T P_N A_{N-1} x_{N-1} = 0
$$

ここから$u_{N-1}$を求めると、それが$J_{N-1}$が最小となる最適入力$u_{N-1}^*$となる。$R_{N-1} + B_{N-1}^T P_N B_{N-1}$は正定値であるため逆行列が存在する。したがって、

$$
\begin{aligned}
u_{N-1} &= - \left\{ (R_{N-1} + B_{N-1}^T P_N B_{N-1} )^{-1} B_{N-1}^T P_N A_{N-1} \right\} x_{N-1}
\end{aligned}
$$

となる。この最小化によって、状態$x_{N-1}$に対する最適入力を与える最適制御則が得られる。

$$
\boxed{
u_{N-1}^* = - K_{N-1} x_{N-1}, \quad K_{N-1} = (R_{N-1} + B_{N-1}^T P_N B_{N-1} )^{-1} B_{N-1}^T P_N A_{N-1}
}
$$

最適入力が$u_{N-1}^* = -K_{N-1} x_{N-1}$ と現在状態に比例する形で得られるため、$K_{N-1}$は状態フィードバックゲインである。

この$u_{N-1}^*$を $J_{N-1}$ に代入する。これより$u_{N-1}^*$は状態$x_{N-1}$によって置き換えられ、最小コストを表す$V_{N-1}$は$x_{N-1}$の二次形式となる。

$$
\begin{aligned}
V_{N-1}(x_{N-1}) &= J_{N-1}(x_{N-1}, u_{N-1}^*) \\
&=  \frac{1}{2}x_{N-1}^T Q_{N-1} x_{N-1} + \frac{1}{2} (K_{N-1} x_{N-1})^T R_{N-1} (K_{N-1} x_{N-1}) \\
&+ \frac{1}{2} \left( ( A_{N-1} - B_{N-1} K_{N-1})  x_{N-1}) \right)^T P_N \left( ( A_{N-1} - B_{N-1} K_{N-1})  x_{N-1}) \right) \\
=&  \frac{1}{2}x_{N-1}^T \Big[ Q_{N-1} + K_{N-1}^T R_{N-1} K_{N-1} \\
& \qquad + \left(  A_{N-1} - B_{N-1} K_{N-1} \right)^T P_N \left(  A_{N-1} - B_{N-1} K_{N-1} \right) \Big]  x_{N-1}
\end{aligned}
$$

上式の$[ \ ]$括弧の中を$P_{N-1}$とおく。

$$
\boxed{
\begin{aligned}
P_{N-1} &= Q_{N-1} + K_{N-1}^T R_{N-1} K_{N-1} \\
& \qquad + \left( A_{N-1} - B_{N-1} K_{N-1} \right)^T P_N \left( A_{N-1} - B_{N-1} K_{N-1} \right) 
\end{aligned}
}
$$

この$P_{N-1}$を用いて、$V_{N-1}$は次のような二次形式になる。

$$
V_{N-1}(x_{N-1}) = \frac{1}{2} x_{N-1}^T P_{N-1} x_{N-1}
$$

よって、$V_{N-1}$は、ひとつ前の $P_N$ から $K_{N-1}$ を計算し、$K_{N-1}$を用いた最適入力 $u_{N-1}^*$ により $P_{N-1}$ を構成して$V_{N-1}$を求めた。最適入力$u_{N-1}^*$は計算上$P_{N-1}$を計算するだけならば式に現れないことに注意する。

$$
P_N \rightarrow K_{N-1} \rightarrow P_{N-1} \rightarrow V_{N-1}
$$


同じく、$N-2$、$N-3$と順番に$ $P_{N-1} \rightarrow K_{N-2} \rightarrow P_{N-2} \rightarrow K_{N-3} \rightarrow P_{N-3} \rightarrow \cdots$と最適入力を代入しながら計算していくことで、以下のように価値関数を$x_k$に関する二次形式とすることが出来る。

$$
\begin{aligned}
V_{N-2}(x_{N-2}) &= \frac{1}{2} x_{N-2}^T P_{N-2} x_{N-2} \\
V_{N-3}(x_{N-3}) &= \frac{1}{2} x_{N-3}^T P_{N-3} x_{N-3} \\
& \ \vdots
\end{aligned}
$$

$k$時点の$K_k$と$P_k$の計算式を以下に示す。それぞれの計算には状態も入力も含まれておらず、モデルパラメータの$A_k,B_k$、コストの重み行列 $Q_k, R_k$ で計算が行われている。計算を開始する終端では$P_N=Q_N$であることに注意する。

$$
\boxed{
\begin{aligned}
K_{k} &= (R_{k} + B_{k}^T P_{k+1} B_{k} )^{-1} B_{k}^T P_{k+1} A_{k} \\
P_{k} &= Q_{k} + K_{k}^T R_{k} K_{k} \\
& \qquad + \left( A_{k} - B_{k} K_{k} \right)^T P_{k+1} \left(  A_{k} - B_{k} K_{k}  \right) 
\end{aligned}
}
$$

$K_{k}$ は状態フィードバックゲインであり、$P_{k+1}$から$P_{k}$を終端から逆向きに計算する処理をRiccati再帰と呼ぶ。

このRiccati再帰により、状態フィードバックゲインの列を状態と入力によらず計算することが出来る。


最適入力を求める最適制御則は$u_k^* = -K_k x_k$ であるため、このRiccati再帰により状態フィードバックゲイン$K$の列が求まっていれば、
$x_k$を与えることで、最適入力$u_k^*$が求まる、という関係である。

#### 各時刻の最適入力の一意性

各時刻$k$で最適入力$u_k^*$が一意に決まることを $u_k$に関するのHessian $H_k$が正定値であることから示す。
まず、$P_N = Q_N \succeq 0$ であり、ひとつ前の$P_{N-1}$は$Q_{N-1} \succeq 0, R_{N-1} \succ 0$である。一般に$M \succeq 0$の行列でれば、任意の行列$C$に対して、$C^T M C \succeq 0$である。$M \succ 0$は$M \succeq 0$ でもあるため、この性質を適用できる。よって、$P_{N-1}$は半正定値行列の和であるため、$P_{N-1} \succeq 0$ である。

$$
\begin{aligned}
P_{N-1} &= Q_{N-1} + K_{N-1}^T R_{N-1} K_{N-1} \\
& \qquad + \left( A_{N-1} - B_{N-1} K_{N-1} \right)^T P_{N} \left(  A_{N-1} - B_{N-1} K_{N-1}  \right) 
\end{aligned}
$$

この時$P_{N-1}$を用いて次の$J_{N-2}$の$Hessian $H_{N-2}$は以下となる。$R_{N-2} \succ 0$であるため、$H_{N-2} \succ 0$となるため最適入力$u_{N-2}^*$は一意に決まる。

$$
H_{N-2} = R_{N-2} + B_{N-2}^T P_{N-1} B_{N-2}
$$

これを一般時刻$k$として考えた場合、$Q_k \succeq 0, R_k \succ 0, P_{k+1} \succeq 0$ より、$P_k \succeq 0$である。

$$
\begin{aligned}
P_{k} &= Q_{k} + K_{k}^T R_{k} K_{k} \\
& \qquad + \left( A_{k} - B_{k} K_{k} \right)^T P_{k+1} \left(  A_{k} - B_{k} K_{k}  \right) 
\end{aligned}
$$

そして、$J_{k-1}$のHessian $H_{k-1}$は$R_{k-1} \succ 0, P_k \succeq 0$ より、$H_{k-1} \succ 0$ となり、最適入力$u_{k-1}^*$が一意に決まる。

$$
H_{k-1} = R_{k-1} + B_{k-1}^T P_{k} B_{k-1}
$$

Riccati再帰中に$H$と$u^*$を組み込むと以下のようなイメージの流れになる。

$$
\begin{aligned}
P_N \succeq 0 & \Rightarrow H_{N-1} \succ 0 \Rightarrow \underset{u_{N-1}}{\arg \min} J_{N-1}(x_{N-1}, u_{N-1}) \Rightarrow K_{N-1} \Rightarrow \\
P_{N-1} \succeq 0 & \Rightarrow H_{N-2} \succ 0 \Rightarrow \underset{u_{N-2}}{\arg \min} J_{N-2}(x_{N-2}, u_{N-2}) \Rightarrow K_{N-2}  \Rightarrow\\
& \vdots \\
P_{k} \succeq 0 & \Rightarrow H_{k-1} \succ 0 \Rightarrow \underset{u_{k-1}}{\arg \min} J_{k-1}(x_{k-1}, u_{k-1}) \Rightarrow K_{k-1}  \Rightarrow\\
& \vdots \\
P_{1} \succeq 0 & \Rightarrow H_{0} \succ 0 \Rightarrow \underset{u_{0}}{\arg \min} J_{0}(x_{0}, u_{0}) \Rightarrow K_{0}
\end{aligned}
$$


## まとめ

以上より、有限時間LQRでは、終端価値関数

$$
V_N(x_N) = \frac{1}{2} x_N^T Q_N x_N
$$

より、

$$
P_N = Q_N
$$

を出発点として、Bellman方程式に基づき、

$$
K_{k} = (R_{k} + B_{k}^T P_{k+1} B_{k} )^{-1} B_{k}^T P_{k+1} A_{k} 
$$

および

$$
P_{k} = Q_{k} + K_{k}^T R_{k} K_{k} + \left( A_{k} - B_{k} K_{k} \right)^T P_{k+1} \left(  A_{k} - B_{k} K_{k}  \right) 
$$

を $k=N-1 , \cdots , 0$ の順番で計算する。

この終端から逆向きの計算をRiccati再帰、またはbackward pass と呼ぶ。

backward pass では、具体的な状態$x_k$や入力$u_k$は計算に用いずに、モデル $A_k, B_k$、コスト重み$Q_k, R_k, Q_N$ から、価値関数を表す $P_k$ と最適状態フィードバックゲイン$K_k$ の列を求める。

$$
P_N \rightarrow K_{N-1}, P_{N-1} \rightarrow K_{N-2}, P_{N-2} \rightarrow \cdots \rightarrow K_0, P_0
$$

これより、以下の各時刻の最適制御則が得られる。

$$
u_k^* (x_k) = -K_k x_k
$$



backward passの後に、具体的な初期状態$x_0$から $u_k^* = -K_k x_k$ を適用して順方向にrolloutすることで、最適状態軌道 $X^*$ と最適入力列 $U^*$を求めることが出来る。これをforward passと呼ぶ。

### Pythonによるbackward passの実装

簡単のため、パラメータは変化しないとする。計算されるパラメータの正しさではなく、実装の方法について理解を目的とする。

backward passで計算を行ったフィードバックゲイン列を使い、forward pass で具体的な$x_0$から最適状態軌道$X^*$と最適入力列$U^*$を求める。

その後、価値関数$V_0$とコスト $J(x_0, U^*)$が一致していることを確認する。

価値関数$V_0$は時刻$0$から$N$までのコストの最小値であり、以下の式で表される。

$$
V_0(x_0) = \frac{1}{2} x_0^T P_0 x_0
$$

コスト$J(x_0, U^*)$は以下の式で表される。以下では状態の初期値は$x_0$だが、最適入力による状態軌道を$x_k^*$ で表している。

$$
\begin{aligned}
J(x_0, U^*) &= \phi(x_N^*) + \sum_{k=0}^{N-1} \ell_k ( x_k^*, u_k^*) \\
&= \frac{1}{2} {x_N^*}^T Q_N {x_N^*} + \sum_{k=0}^{N-1}\left( \frac{1}{2} {x_k^*}^T \ Q \ {x_k^*} + \frac{1}{2} {u_k^*}^T \ R \ {u_k^*} \right)
\end{aligned}
$$



In [17]:
from dataclasses import dataclass, field
import numpy as np
from scipy.linalg import solve


@dataclass
class ModelParameters:
    N :int = 10 # 有限時間のステップ数

    # 状態次元 nx = 2
    # 入力次元 nu = 1

    # パラメータは変化しないと仮定する

    ## 重み行列
    R: np.ndarray = field(
        default_factory=lambda: np.array([[1.0]])
    )

    Q: np.ndarray = field(
        default_factory=lambda: np.array([
            [2.0, 0.0],
            [0.0, 0.1]
        ])
    )

    Qn : np.ndarray = field(
        default_factory=lambda: np.array([
            [3.0, 0.0],
            [0.0, 1.0]
        ])
    )

    ## モデルパラメーター
    A : np.ndarray = field(
        default_factory=lambda: np.array([
            [0.5 , 0.1],
            [0.1, -2.0]
        ])
    )

    B: np.ndarray = field(
        default_factory=lambda: np.array([
            [0.0],
            [1.0]
        ])
    )

def backward_pass(params : ModelParameters):

    nx = params.A.shape[0]
    nu = params.B.shape[1]

    # K_k: (nu, nx)
    K_array = np.zeros((params.N, nu, nx))

    # P_0 , ... P_N
    P_array = np.zeros((params.N+1, nx, nx))
    #終端条件 P_N = Q_N
    P_array[params.N] = params.Qn

    for k in range(params.N -1 , -1, -1):
        P_next = P_array[k+1]

        H = params.R + params.B.T @ P_next @ params.B
        G = params.B.T @ P_next @ params.A

        # H K_k = G を解く
        K = solve(H, G, assume_a="pos") # pos : で正定値行列であることを指定

        # closed_loop : (A - B K)
        closed_loop = params.A - params.B@K

        P = (
            params.Q 
            + K.T @ params.R @ K 
            + closed_loop.T @ P_next @ closed_loop
        )

        K_array[k] = K
        P_array[k] = P

    return K_array, P_array

def forward_pass(params, K_array , x0):

    nx = params.A.shape[0]
    nu = params.B.shape[1]

    X = np.zeros((params.N + 1,nx))
    U = np.zeros((params.N, nu))

    X[0] = x0
    for k in range(params.N):
        U[k] = - K_array[k] @ X[k]
        X[k+1] = params.A @ X[k] + params.B @ U[k]

    return X, U

def calculate_J(params:ModelParameters, X, U):

    phi_N = 0.5 * X[-1].T @ params.Qn @ X[-1]

    el_sum = 0

    for i in range(params.N):
        el_sum += (
            0.5 * X[i].T @ params.Q @ X[i] 
            + 0.5 * U[i].T @ params.R @ U[i]
        )

    J = phi_N + el_sum

    return J

def main():

    params = ModelParameters()

    feedback_gain_array, P_array = backward_pass(params=params)

    for k , K in enumerate(feedback_gain_array):
        print(f"K[{k}] = {K}")

    print(f"\nP[0] = \n{P_array[0]}")

    # 状態の初期値
    x0 = np.array([10,2])

    X, U = forward_pass(params, feedback_gain_array, x0)

    J = calculate_J(params, X, U)

    V0 = 0.5 * X[0].T @ P_array[0] @ X[0]

    print(f"J : {J}, V0 : {V0}")

    print(f"一致している? : {np.isclose(J, V0)}")


if __name__ ==  "__main__":
    main()

K[0] = [[ 0.07424637 -1.52057704]]
K[1] = [[ 0.07424595 -1.52057201]]
K[2] = [[ 0.07424696 -1.52055033]]
K[3] = [[ 0.07423955 -1.52045708]]
K[4] = [[ 0.07425567 -1.52005574]]
K[5] = [[ 0.07412494 -1.51833178]]
K[6] = [[ 0.07437948 -1.51095233]]
K[7] = [[ 0.07213168 -1.47996958]]
K[8] = [[ 0.07603834 -1.35942492]]
K[9] = [[ 0.05 -1.  ]]

P[0] = 
[[ 2.67631388 -0.01471489]
 [-0.01471489  3.16862265]]
J : 139.85864145436094, V0 : 139.8586414543609
一致している? : True
